In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load and describe the data
* Load the data (in wide format) and convert to long format
* Merge the 2 time series and filter to the time window of interest (1992 - 2024)
* Drop regional/income-groups with now regions

In [4]:
LAST_RECENT_YEAR = 2024

In [5]:
# ── 1. Load ────────────────────────────────────────────────────────────────────
co2_wide = pd.read_csv("data/API_EN.GHG.CO2.PC.CE.AR5_DS2_en_csv_v2_610.csv", skiprows=4)
gdp_wide = pd.read_csv("data/API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_35.csv", skiprows=4)
meta     = pd.read_csv("data/Metadata_Country_API_EN.GHG.CO2.PC.CE.AR5_DS2_en_csv_v2_610.csv")

# ── 2. Wide → Long ─────────────────────────────────────────────────────────────
id_vars = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
year_cols = [c for c in co2_wide.columns if c.isdigit() and int(c) <= LAST_RECENT_YEAR]

co2 = (co2_wide
       .melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name="co2_pc")
       .assign(year=lambda x: x["year"].astype(int)))

gdp = (gdp_wide
       .melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name="gdp_pc")
       .assign(year=lambda x: x["year"].astype(int)))

# ── 3. Merge + filter to assignment window (1992–latest) ──────────────────────
df = (co2[["Country Code", "Country Name", "year", "co2_pc"]]
      .merge(gdp[["Country Code", "year", "gdp_pc"]], on=["Country Code", "year"])
      .merge(meta[["Country Code", "Region", "IncomeGroup"]], on="Country Code")
      .query("year >= 1992")
      .rename(columns={"Country Name": "country", "Country Code": "iso3"}))

# drop aggregates (World Bank includes regional/income-group rows with no Region)
df = df[df["Region"].notna()].copy()

In [6]:
# ── 4. Summary statistics ──────────────────────────────────────────────────────
print(f"Countries: {df['iso3'].nunique()}")
print(f"Years:     {df['year'].min()} – {df['year'].max()}")
print(f"Obs:       {len(df):,}")
print(f"\nMissing co2_pc: {df['co2_pc'].isna().mean():.2%}")
print(f"Missing gdp_pc: {df['gdp_pc'].isna().mean():.2%}")
print("\nSummary statistics:")
print(df[["co2_pc", "gdp_pc"]].describe().round(2))

Countries: 217
Years:     1992 – 2024
Obs:       7,161

Missing co2_pc: 6.45%
Missing gdp_pc: 9.80%

Summary statistics:
        co2_pc     gdp_pc
count  6699.00    6459.00
mean      4.85   17912.89
std       9.40   21596.41
min       0.00     254.39
25%       0.52    3259.44
50%       2.19    9469.67
75%       6.13   24781.25
max     202.87  180939.44


# Dropping countries with low data coverage

In [7]:
total_years = df["year"].nunique()

coverage = (df.groupby(["iso3", "country"])[["co2_pc", "gdp_pc"]]
              .apply(lambda x: x.notna().all(axis=1).sum())
              .rename("both_obs")
              .reset_index()
              .assign(coverage_pct=lambda x: x["both_obs"] / total_years * 100))

print(coverage["coverage_pct"].describe().round(1))
print(f"\nCountries below 80% coverage: {(coverage['coverage_pct'] < 80).sum()}")
print(f"Countries below 50% coverage: {(coverage['coverage_pct'] < 50).sum()}")

count    217.0
mean      87.2
std       31.8
min        0.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: coverage_pct, dtype: float64

Countries below 80% coverage: 33
Countries below 50% coverage: 25


In [8]:
# ── Drop countries with below 80% coverage ────────────────────────────────────────────────
THRESHOLD = 50

good_coverage = coverage.loc[coverage["coverage_pct"] >= THRESHOLD, "iso3"]
df_clean = df[df["iso3"].isin(good_coverage)].copy()

dropped = coverage[coverage["coverage_pct"] < THRESHOLD][["iso3", "country", "coverage_pct"]].sort_values("coverage_pct")
print(f"\nDropped {len(dropped)} countries:")
print(dropped.to_string(index=False))
print(f"\nRemaining: {df_clean['iso3'].nunique()} countries, {len(df_clean):,} obs")


Dropped 25 countries:
iso3                   country  coverage_pct
 AND                   Andorra      0.000000
 ASM            American Samoa      0.000000
 CHI           Channel Islands      0.000000
 CUB                      Cuba      0.000000
 CUW                   Curacao      0.000000
 GIB                 Gibraltar      0.000000
 GUM                      Guam      0.000000
 IMN               Isle of Man      0.000000
 MNE                Montenegro      0.000000
 LIE             Liechtenstein      0.000000
 MAF  St. Martin (French part)      0.000000
 MCO                    Monaco      0.000000
 NCL             New Caledonia      0.000000
 MNP  Northern Mariana Islands      0.000000
 PRK Korea, Dem. People's Rep.      0.000000
 PSE        West Bank and Gaza      0.000000
 SSD               South Sudan      0.000000
 PYF          French Polynesia      0.000000
 SMR                San Marino      0.000000
 SRB                    Serbia      0.000000
 VGB    British Virgin Islands  

In [9]:
# ── Cleaned sample: Summary statistics ──────────────────────────────────────────────────────
print(f"Countries: {df_clean['iso3'].nunique()}")
print(f"Years:     {df_clean['year'].min()} – {df_clean['year'].max()}")
print(f"Obs:       {len(df_clean):,}")
print(f"\nMissing co2_pc: {df_clean['co2_pc'].isna().mean():.2%}")
print(f"Missing gdp_pc: {df_clean['gdp_pc'].isna().mean():.2%}")
print("\nSummary statistics:")
print(df_clean[["co2_pc", "gdp_pc"]].describe().round(2))

Countries: 192
Years:     1992 – 2024
Obs:       6,336

Missing co2_pc: 0.00%
Missing gdp_pc: 1.74%

Summary statistics:
        co2_pc     gdp_pc
count  6336.00    6226.00
mean      4.92   17683.99
std       9.59   21614.46
min       0.00     254.39
25%       0.52    3170.66
50%       2.20    9305.81
75%       6.23   24276.91
max     202.87  180939.44


# Estimate and interpret coefficients for each model

## 1. Cross-section OLS for year 2005

In [10]:
# create a helper function for cross section OLS of a year
import statsmodels.formula.api as smf

def cross_section_ols(df, year):
    d = (df[df["year"] == year][["iso3", "country", "co2_pc", "gdp_pc"]]
           .dropna()
           .query("co2_pc > 0 and gdp_pc > 0")    # only include countries with both positive GDP and CO2 per capita
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]))
        )

    model = smf.ols("log_co2 ~ log_gdp", data=d).fit(cov_type="HC3")

    # manually build results table instead of calling summary()
    results_table = pd.DataFrame({
        "coef":    model.params,
        "HC3_se":  model.bse,
        "t":       model.tvalues,
        "p-value": model.pvalues,
        "CI_low":  model.conf_int()[0],
        "CI_high": model.conf_int()[1],
    }).round(4)

    print(f"\n── Cross-section OLS: {year} ── N={len(d)} countries ──")
    print(f"R²: {model.rsquared:.3f}")
    print(results_table)

    return model

In [11]:
# run the model for 2005
model_2005 = cross_section_ols(df_clean, 2005)


── Cross-section OLS: 2005 ── N=187 countries ──
R²: 0.640
              coef  HC3_se        t  p-value   CI_low  CI_high
Intercept -10.4256  0.5964 -17.4806      0.0 -11.5946  -9.2567
log_gdp     1.2180  0.0679  17.9320      0.0   1.0849   1.3512


## 2. Cross-section OLS for latest available year

In [12]:
# run the model for 2024
model_2022 = cross_section_ols(df_clean, df_clean['year'].max())


── Cross-section OLS: 2024 ── N=179 countries ──
R²: 0.634
             coef  HC3_se        t  p-value   CI_low  CI_high
Intercept -9.7603  0.6405 -15.2393      0.0 -11.0156  -8.5050
log_gdp    1.0599  0.0646  16.4062      0.0   0.9333   1.1865


## 3. First difference model, with time trend, no lags

In [13]:
from linearmodels.panel import PanelOLS

def first_diff_model(df, lags=0):
    """
    First difference model using PanelOLS.
    TimeEffects replaces C(year) dummies.
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query("co2_pc > 0 and gdp_pc > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]))
           .sort_values(["iso3", "year"]))

    # ── first differences within each country ─────────────────────────────────
    d["dlog_co2"] = d.groupby("iso3")["log_co2"].diff()
    d["dlog_gdp"] = d.groupby("iso3")["log_gdp"].diff()

    # ── lags ──────────────────────────────────────────────────────────────────
    formula_vars = "dlog_gdp"
    if lags > 0:
        for lag in range(1, lags + 1):
            d[f"dlog_gdp_lag{lag}"] = d.groupby("iso3")["dlog_gdp"].shift(lag)
            formula_vars += f" + dlog_gdp_lag{lag}"

    # ── dropna + set MultiIndex (required by linearmodels) ────────────────────
    drop_cols = ["dlog_co2", "dlog_gdp"] + [f"dlog_gdp_lag{lag}" for lag in range(1, lags+1)]
    d = (d.dropna(subset=drop_cols)
          .set_index(["iso3", "year"]))  # PanelOLS requires MultiIndex

    model = PanelOLS.from_formula(
        f"dlog_co2 ~ {formula_vars} + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print(f"\n── FD Model (PanelOLS) | lags={lags} ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [14]:
model_fd0 = first_diff_model(df_clean, lags=0)


── FD Model (PanelOLS) | lags=0 ──
N obs: 5,971  |  N countries: 190
R² (within): 0.023
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
dlog_gdp       0.3195     0.0481     6.6417     0.0000      0.2252      0.4138


## 4. First difference model, with time trend, 2 year lags

In [15]:
model_fd2 = first_diff_model(df_clean, lags=2)


── FD Model (PanelOLS) | lags=2 ──
N obs: 5,591  |  N countries: 190
R² (within): 0.019
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
dlog_gdp          0.2708     0.0482     5.6138     0.0000      0.1762      0.3653
dlog_gdp_lag1     0.0191     0.0472     0.4057     0.6850     -0.0734      0.1116
dlog_gdp_lag2     0.0516     0.0214     2.4120     0.0159      0.0097      0.0935


## 5. First difference model, with time trend, 6 year lags

In [16]:
model_fd6 = first_diff_model(df_clean, lags=6)


── FD Model (PanelOLS) | lags=6 ──
N obs: 4,831  |  N countries: 190
R² (within): 0.019
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
dlog_gdp          0.2687     0.0520     5.1675     0.0000      0.1667      0.3706
dlog_gdp_lag1     0.0381     0.0495     0.7709     0.4408     -0.0588      0.1351
dlog_gdp_lag2     0.0202     0.0246     0.8199     0.4123     -0.0281      0.0685
dlog_gdp_lag3     0.0232     0.0344     0.6737     0.5005     -0.0443      0.0906
dlog_gdp_lag4     0.0410     0.0342     1.2006     0.2300     -0.0260      0.1081
dlog_gdp_lag5    -0.0366     0.0370    -0.9896     0.3224     -0.1091      0.0359
dlog_gdp_lag6     0.0457     0.0322     1.4172     0.1565     -0.0175      0.1088


## 6. Fixed effects model with time and country fixed effects

In [17]:
# from linearmodels.panel import PanelOLS

def fixed_effects_model(df):
    """
    Fixed effects model with time and country fixed effects.
    Equivalent to: log_co2 ~ log_gdp + EntityEffects + TimeEffects
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query("co2_pc > 0 and gdp_pc > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]))
           .dropna(subset=["log_co2", "log_gdp"])
           .set_index(["iso3", "year"]))  # linearmodels requires MultiIndex

    model = PanelOLS.from_formula(
        "log_co2 ~ log_gdp + EntityEffects + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print("\n── Fixed Effects Model | Country + Time FE ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [18]:
model_fe = fixed_effects_model(df_clean)


── Fixed Effects Model | Country + Time FE ──
N obs: 6,161  |  N countries: 190
R² (within): 0.069
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
log_gdp        0.4414     0.0747     5.9105     0.0000      0.2950      0.5878


# Potential confounder: Electricity power consumption per capita

## Prepare the dataset with the confounder variable included

In [19]:
CONFOUNDER = "urb_pct"

In [20]:
# ── Load industry share and join to existing dataset ───────────────────────────────────────────────────
industry_wide = pd.read_csv("data/API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_249.csv", skiprows=4)

id_vars   = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
year_cols = [c for c in industry_wide.columns if c.isdigit() and int(c) <= 2024]

industry = (industry_wide
            .melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name=CONFOUNDER)
            .assign(year=lambda x: x["year"].astype(int))
            [["Country Code", "year", CONFOUNDER]]
            .rename(columns={"Country Code": "iso3"}))

# ── Join to existing df_clean ──────────────────────────────────────────────────
df_clean_ind = df_clean.merge(industry, on=["iso3", "year"], how="left")

# ── Check ──────────────────────────────────────────────────────────────────────
print(f"Countries: {df_clean_ind['iso3'].nunique()}")  # should match df_clean
print(f"Obs:       {len(df_clean_ind):,}")              # should match df_clean
print(f"Missing {CONFOUNDER}: {df_clean_ind[CONFOUNDER].isna().sum():,}")
print()

# ── Cleaned sample: Summary statistics ──────────────────────────────────────────────────────
print(f"Countries: {df_clean_ind['iso3'].nunique()}")
print(f"Years:     {df_clean_ind['year'].min()} – {df_clean['year'].max()}")
print(f"Obs:       {len(df_clean_ind):,}")
print(f"\nMissing co2_pc: {df_clean_ind['co2_pc'].isna().mean():.2%}")
print(f"Missing gdp_pc: {df_clean_ind['gdp_pc'].isna().mean():.2%}")
print(f"Missing {CONFOUNDER}: {df_clean_ind[CONFOUNDER].isna().mean():.2%}")
print("\nSummary statistics:")
print(df_clean_ind[["co2_pc", "gdp_pc", CONFOUNDER]].describe().round(2))

Countries: 192
Obs:       6,336
Missing urb_pct: 0

Countries: 192
Years:     1992 – 2024
Obs:       6,336

Missing co2_pc: 0.00%
Missing gdp_pc: 1.74%
Missing urb_pct: 0.00%

Summary statistics:
        co2_pc     gdp_pc  urb_pct
count  6336.00    6226.00  6336.00
mean      4.92   17683.99    56.73
std       9.59   21614.46    23.77
min       0.00     254.39     5.66
25%       0.52    3170.66    36.32
50%       2.20    9305.81    57.12
75%       6.23   24276.91    75.94
max     202.87  180939.44   100.00


In [21]:
df_clean_ind.head()

,iso3,country,year,co2_pc,gdp_pc,Region,IncomeGroup,urb_pct
0,ABW,Aruba,1992,3.634519,23889.045018,Latin America & Caribbean,High income,65.390589
1,AFG,Afghanistan,1992,0.136358,NaN,Middle East & North Africa,Low income,17.488203
2,AGO,Angola,1992,0.975473,3486.206867,Sub-Saharan Africa,Lower middle income,40.520577
3,ALB,Albania,1992,0.727802,1899.258869,Europe & Central Asia,Upper middle income,36.718428
4,ARE,United Arab Emirates,1992,29.565193,87509.043492,Middle East & North Africa,High income,78.441374


## 1. Cross-sectional OLS for 2005

In [22]:
# create a helper function for cross section OLS of a year
import statsmodels.formula.api as smf

def cross_section_ols(df, year):
    d = (df[df["year"] == year][["iso3", "country", "co2_pc", "gdp_pc", CONFOUNDER]]
           .dropna()
           .query(f"co2_pc > 0 and gdp_pc > 0 and {CONFOUNDER} > 0")    # only include countries with both positive GDP and CO2 per capita
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]),
                   )
        )

    model = smf.ols(f"log_co2 ~ log_gdp + {CONFOUNDER}", data=d).fit(cov_type="HC3")

    # manually build results table instead of calling summary()
    results_table = pd.DataFrame({
        "coef":    model.params,
        "HC3_se":  model.bse,
        "t":       model.tvalues,
        "p-value": model.pvalues,
        "CI_low":  model.conf_int()[0],
        "CI_high": model.conf_int()[1],
    }).round(4)

    print(f"\n── Cross-section OLS: {year} ── N={len(d)} countries ──")
    print(f"R²: {model.rsquared:.3f}")
    print(results_table)

    return model

In [23]:
model_2005_ind = cross_section_ols(df_clean_ind, 2005)


── Cross-section OLS: 2005 ── N=187 countries ──
R²: 0.640
              coef  HC3_se        t  p-value   CI_low  CI_high
Intercept -10.2353  0.7378 -13.8730    0.000 -11.6813  -8.7892
log_gdp     1.1822  0.1148  10.2963    0.000   0.9572   1.4073
urb_pct     0.0024  0.0076   0.3094    0.757  -0.0126   0.0173


## 4. First difference model, with time trend, 2 year lags

In [24]:
from linearmodels.panel import PanelOLS

def first_diff_model(df, lags=0):
    """
    First difference model using PanelOLS.
    TimeEffects replaces C(year) dummies.
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query(f"co2_pc > 0 and gdp_pc > 0 and {CONFOUNDER} > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]),
                   )
           .sort_values(["iso3", "year"]))

    # ── first differences within each country ─────────────────────────────────
    d["dlog_co2"] = d.groupby("iso3")["log_co2"].diff()
    d["dlog_gdp"] = d.groupby("iso3")["log_gdp"].diff()
    d[f"d{CONFOUNDER}"] = d.groupby("iso3")[f"{CONFOUNDER}"].diff()

    # ── lags ──────────────────────────────────────────────────────────────────
    formula_vars = f"dlog_gdp + d{CONFOUNDER}"
    if lags > 0:
        for lag in range(1, lags + 1):
            d[f"dlog_gdp_lag{lag}"] = d.groupby("iso3")["dlog_gdp"].shift(lag)
            formula_vars += f" + dlog_gdp_lag{lag}"

    # ── dropna + set MultiIndex (required by linearmodels) ────────────────────
    drop_cols = ["dlog_co2", "dlog_gdp", f"d{CONFOUNDER}"] + [f"dlog_gdp_lag{lag}" for lag in range(1, lags+1)]
    d = (d.dropna(subset=drop_cols)
          .set_index(["iso3", "year"]))  # PanelOLS requires MultiIndex

    model = PanelOLS.from_formula(
        f"dlog_co2 ~ {formula_vars} + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print(f"\n── FD Model (PanelOLS) | lags={lags} ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [25]:
model_fd2_ind = first_diff_model(df_clean_ind, lags=2)


── FD Model (PanelOLS) | lags=2 ──
N obs: 5,591  |  N countries: 190
R² (within): 0.021
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
dlog_gdp          0.2701     0.0481     5.6130     0.0000      0.1758      0.3644
durb_pct          0.0113     0.0037     3.0646     0.0022      0.0041      0.0186
dlog_gdp_lag1     0.0173     0.0472     0.3664     0.7141     -0.0753      0.1099
dlog_gdp_lag2     0.0483     0.0212     2.2769     0.0228      0.0067      0.0900


## 6. Fixed effect model with time and country fixed effect

In [26]:
# from linearmodels.panel import PanelOLS

def fixed_effects_model(df):
    """
    Fixed effects model with time and country fixed effects.
    Equivalent to: log_co2 ~ log_gdp + EntityEffects + TimeEffects
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query(f"co2_pc > 0 and gdp_pc > 0 and {CONFOUNDER} > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]),
                   )
           .dropna(subset=["log_co2", "log_gdp", CONFOUNDER])
           .set_index(["iso3", "year"]))  # linearmodels requires MultiIndex

    model = PanelOLS.from_formula(
        f"log_co2 ~ log_gdp + {CONFOUNDER} + EntityEffects + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print("\n── Fixed Effects Model | Country + Time FE ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [27]:
model_fe_ind = fixed_effects_model(df_clean_ind)


── Fixed Effects Model | Country + Time FE ──
N obs: 6,161  |  N countries: 190
R² (within): 0.100
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
log_gdp        0.4136     0.0695     5.9517     0.0000      0.2774      0.5498
urb_pct        0.0187     0.0048     3.8879     0.0001      0.0093      0.0281
